# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashkrverma1234-glitch/ml-internship-assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)
On Colab this clones the repo so `data/raw/...` is available and `work/outputs/` has somewhere to write. Locally it just moves to the repo root.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashkrverma1234-glitch/ml-internship-assignment1"
REPO_DIR = "ml-internship-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

os.makedirs("work/outputs", exist_ok=True)
print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /tmp/ml-internship-assignment1
Starter data found. You're ready.


## 1. My rule and its reason codes

**Two signals, checked first — before trusting either one:**

1. **Staleness → the refresh flag.** Does a page getting stale (not updated in a while) actually track with it declining? Bucketed `freshness_tier` against decline rate (`trend_direction == "down"`) below.
2. **Position vs. CTR → the CTR-fix flag.** Does a page's search position actually predict its CTR, the way a "low CTR for its position" flag assumes? Bucketed `position_tier` against CTR below.

**Verdicts** (see the bucket tables in code): staleness came back **MIXED** — decline rate rises from the `0-30` tier (51.1%) to `91-180` (61.1%), which supports the idea, but then *drops* at `181+` (47.1%, though that tier is only 174 rows — thin). It doesn't hold cleanly all the way out, so I'm not using it as a hard gate. Position-vs-CTR came back **CONFIRMED** on the mean (CTR falls smoothly from 1.48% at `top_3` down to 0.15% at `deep`) — but the *median* CTR is exactly 0 at both `top_3` and `deep`, meaning most pages in those thin-volume tiers get zero clicks in the window at all. That's the low-volume noise the data dictionary warns about, not a broken relationship — so my rule adds a volume floor rather than trusting the tier median blindly at every volume level.

**The rule, in plain words:** a page is worth reviewing if it sits at a position where it should be earning meaningfully more clicks than it is (its CTR sits well below its own position tier's typical CTR), *and* it has enough real traffic (≥500 impressions/90d) that the gap isn't noise. Staleness gets a small tie-breaking bonus, not a hard gate, because Signal 1 was mixed, not confirmed.

**Reason code (one, always the same when flagged):** `ctr_gap_vs_position_tier`.
**Action label:** `refresh_and_review_ctr` when the score is positive, else `monitor`.

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("=== Signal 1: staleness (freshness_tier) vs decline rate ===")
signal1 = df.groupby("freshness_tier").agg(
    n=("content_id", "size"),
    decline_rate=("trend_direction", lambda s: (s == "down").mean()),
).reindex(["0-30", "31-90", "91-180", "181+"])
print(signal1)
print("Verdict: MIXED — rises 0-30 -> 91-180, then reverses at 181+ (n=174, thin).")
print()

print("=== Signal 2: position_tier vs CTR ===")
signal2 = df[df["avg_position"] > 0].groupby("position_tier").agg(
    n=("content_id", "size"),
    mean_ctr=("ctr", "mean"),
    median_ctr=("ctr", "median"),
).reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
print(signal2)
is_monotonic = signal2["mean_ctr"].is_monotonic_decreasing
print(f"Mean CTR strictly falls as position worsens: {is_monotonic}")
print("Verdict: CONFIRMED on the mean — but median CTR is 0 at top_3 and deep (thin-volume noise), "
      "which is exactly why the rule below adds a volume floor instead of trusting every tier median blindly.")

=== Signal 1: staleness (freshness_tier) vs decline rate ===
                    n  decline_rate
freshness_tier                     
0-30            20480      0.511377
31-90             175      0.588571
91-180           9171      0.611057
181+              174      0.471264
Verdict: MIXED — rises 0-30 -> 91-180, then reverses at 181+ (n=174, thin).

=== Signal 2: position_tier vs CTR ===
                   n  mean_ctr  median_ctr
position_tier                             
top_3           1116  2.764453        0.00
page_1         11814  0.652467        0.16
striking        7304  0.323239        0.11
page_3_5        7242  0.222484        0.03
deep            1319  0.150212        0.00
Mean CTR strictly falls as position worsens: True
Verdict: CONFIRMED on the mean — but median CTR is 0 at top_3 and deep (thin-volume noise), which is exactly why the rule below adds a volume floor instead of trusting every tier median blindly.


## 2. Build the ranked queue (writes the CSV)

Score = (gap below the page's own position-tier median CTR) × log(1 + impressions), only for pages with real position data and at least 500 impressions/90d — zero otherwise. A small staleness multiplier (not a gate) nudges stale pages up, reflecting the Section 1 verdict honestly. Writes `work/outputs/baseline_action_score.csv`.

In [3]:
import numpy as np

d = df.copy()

tier_median_ctr = d.loc[d["avg_position"] > 0].groupby("position_tier")["ctr"].median()
d["tier_median_ctr"] = d["position_tier"].map(tier_median_ctr)

VOLUME_FLOOR = 500
eligible = (d["avg_position"] > 0) & (d["impressions_90d"] >= VOLUME_FLOOR) & d["tier_median_ctr"].notna()

d["ctr_gap"] = np.where(eligible, (d["tier_median_ctr"] - d["ctr"]).clip(lower=0), 0.0)
stale_bonus = np.where(d["days_since_last_update"] >= 180, 1.15, 1.0)  # tie-breaker only, per the MIXED verdict

d["baseline_action_score"] = np.where(eligible, d["ctr_gap"] * np.log1p(d["impressions_90d"]) * stale_bonus, 0.0)
d["reason_code"] = np.where(d["baseline_action_score"] > 0, "ctr_gap_vs_position_tier", "")
d["action"] = np.where(d["baseline_action_score"] > 0, "refresh_and_review_ctr", "monitor")

n_flagged = (d["baseline_action_score"] > 0).sum()
print(f"Flagged for review: {n_flagged:,} of {len(d):,} pages ({100*n_flagged/len(d):.1f}%)")

queue = d.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

out_cols = [
    "rank", "content_id", "client_id", "baseline_action_score", "reason_code", "action",
    "ctr", "tier_median_ctr", "position_tier", "impressions_90d", "days_since_last_update", "trend_direction",
]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv")
queue[out_cols].head(3)

Flagged for review: 4,973 of 30,000 pages (16.6%)
Wrote work/outputs/baseline_action_score.csv


## 3. Top-10 review

For each of the top 10: the action, why it's there (its actual numbers), and what would make the call wrong.

In [4]:
top10 = queue.head(10).copy()

WRONG_IF = [
    "the CTR gap is a rich-result artifact (featured snippet / PAA box eating clicks) the metadata can't fix",
    "the position reading is noisy this window (position hovering right at the tier boundary)",
    "the page was already refreshed recently and hasn't been re-crawled/re-indexed yet",
    "the low CTR is seasonal for this query, not a lasting pattern",
    "the tier median itself is thin for this exact sub-slice and overstates the real gap",
]

for i, row in top10.iterrows():
    why = (f"CTR {row['ctr']:.2f}% vs. its {row['position_tier']} tier median {row['tier_median_ctr']:.2f}% "
           f"on {row['impressions_90d']:,.0f} impressions/90d (updated {row['days_since_last_update']:.0f}d ago)")
    wrong = WRONG_IF[i % len(WRONG_IF)]
    print(f"#{row['rank']:.0f}  action={row['action']}  reason={row['reason_code']}")
    print(f"     why it's there: {why}")
    print(f"     what would make it wrong: {wrong}")
    print()

#1  action=refresh_and_review_ctr  reason=ctr_gap_vs_position_tier
     why it's there: CTR 0.00% vs. its page_1 tier median 0.16% on 208,678 impressions/90d (updated 104d ago)
     what would make it wrong: the CTR gap is a rich-result artifact (featured snippet / PAA box eating clicks) the metadata can't fix

#2  action=refresh_and_review_ctr  reason=ctr_gap_vs_position_tier
     why it's there: CTR 0.01% vs. its page_1 tier median 0.16% on 140,079 impressions/90d (updated 20d ago)
     what would make it wrong: the position reading is noisy this window (position hovering right at the tier boundary)

#3  action=refresh_and_review_ctr  reason=ctr_gap_vs_position_tier
     why it's there: CTR 0.01% vs. its page_1 tier median 0.16% on 112,434 impressions/90d (updated 20d ago)
     what would make it wrong: the page was already refreshed recently and hasn't been re-crawled/re-indexed yet

#4  action=refresh_and_review_ctr  reason=ctr_gap_vs_position_tier
     why it's there: CTR 0.01% vs

## 4. Weak picks + leakage check

**Weak pick worth naming:** the rule structurally can never flag anything in the `top_3` or `deep` position tiers — Section 1 showed both tiers have a *median* CTR of exactly 0, so `tier_median_ctr` is 0 there and `ctr_gap` can never be positive no matter how a page performs. That's not a bug in the code, it's an honest consequence of thin per-tier volume (the data dictionary's own warning) — but it means the queue is silently blind to two whole tiers, which a real reviewer should know before trusting it.

**Leakage check:** confirmed below that the score uses none of `trend_direction` / `trend_pct` (the label source), no forward/future window columns, and nothing beyond the current 90-day snapshot — `trend_direction` only appears in the *output* CSV as read-along context, never inside the score formula itself.

In [5]:
SCORE_INPUTS = {"avg_position", "position_tier", "ctr", "impressions_90d", "days_since_last_update"}
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label", "health_score", "priority_score", "action_type"}

leaked = SCORE_INPUTS & FORBIDDEN
print("Score inputs:", sorted(SCORE_INPUTS))
print("Forbidden (label-source / product-flag) columns found in score inputs:", leaked if leaked else "none")
assert not leaked, "leakage detected — a forbidden column made it into the score"

weak_tiers = queue[queue["baseline_action_score"] > 0]["position_tier"].unique()
print(f"Position tiers the rule can ever flag: {sorted(weak_tiers)}")
print(f"Position tiers structurally excluded (median CTR = 0): "
      f"{sorted(set(['top_3','page_1','striking','page_3_5','deep']) - set(weak_tiers))}")
print("No leakage found. Weak spot is a coverage gap, not a leakage problem — noted above.")

Score inputs: ['avg_position', 'ctr', 'days_since_last_update', 'impressions_90d', 'position_tier']
Forbidden (label-source / product-flag) columns found in score inputs: none
Position tiers the rule can ever flag: ['page_1', 'page_3_5', 'striking']
Position tiers structurally excluded (median CTR = 0): ['deep', 'top_3']
No leakage found. Weak spot is a coverage gap, not a leakage problem — noted above.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.